In [1]:
from pathlib import Path
print(Path.cwd())

C:\Users\karol\Bakalarka26\prakticka_cast\notebooks\03_geo


In [2]:
from datetime import datetime
from pathlib import Path
import platform, sys
import sys
sys.path.append(str(Path.cwd().parents[1]))  # pridá koreň projektu (prakticka_cast) na sys.path


DOMAIN = "geo"             # "ts" | "corr" | "geo"
DATASET_NAME = "osm"     # napr. "noaa_demo" | "wdi_demo" | "osm_sk_demo"
LIBRARY = "geopandas"        # "matplotlib" | "seaborn" | "plotly" | "geopandas_matplotlib"
TASK_ID = "geograficke_data"
N_RUNS = 10
SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")


In [3]:
import os, time, json, csv
import numpy as np, pandas as pd, statistics as stats
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import plotly
import plotly.express as px, plotly.graph_objects as go


try:
    import geopandas as gpd
except Exception:
    gpd = None
try:
    import yaml
except Exception:
    yaml = None

plt.rcParams.update({
    "figure.dpi": 120, "figure.figsize": (8, 4.5),
    "axes.titlesize": 12, "axes.labelsize": 10,
    "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "font.family": "DejaVu Sans"
})
sns.set_theme(style="whitegrid")
px.defaults.template = "plotly_white"
px.defaults.width = 800
px.defaults.height = 450

In [4]:
from pathlib import Path
import plotly.io as pio

PROJECT_ROOT = Path.cwd().parents[1]
OUT_ROOT = PROJECT_ROOT / "outputs"

FIG_DIR = OUT_ROOT / "figures"
HTML_DIR = OUT_ROOT / "html"
METRICS_DIR = OUT_ROOT / "metrics"
TABLES_DIR = OUT_ROOT / "tables"

CFG_PLOTS = {"dpi": 120, "svg": True}

for d in [FIG_DIR, HTML_DIR, METRICS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOMAIN_MAP = {"ts": "time_series", "corr": "correlations", "geo": "geo"}
DOMAIN_DIR = DOMAIN_MAP.get(DOMAIN, DOMAIN)
LIB_DIR = LIBRARY.replace("_", "")

FIG_SUBDIR = FIG_DIR / DOMAIN_DIR / LIB_DIR
HTML_SUBDIR = HTML_DIR / DOMAIN_DIR

FIG_SUBDIR.mkdir(parents=True, exist_ok=True)
HTML_SUBDIR.mkdir(parents=True, exist_ok=True)

# pre Plotly geo export PNG/SVG
PLOTLY_TOPOJSON_DIR = PROJECT_ROOT / "assets" / "plotly_topojson"
if PLOTLY_TOPOJSON_DIR.exists():
    pio.defaults.topojson = str(PLOTLY_TOPOJSON_DIR.resolve())


def file_basename(prefix: str, variant: str, ext: str) -> Path:
    name = f"{DOMAIN}_{LIBRARY}_{DATASET_NAME}_{TASK_ID}_{variant}_{TIMESTAMP}.{ext}"
    return (HTML_SUBDIR if ext.lower() == "html" else FIG_SUBDIR) / name

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = (time.perf_counter() - self.t0) * 1000.0

def file_size_kb(path: Path):
    try:
        return os.path.getsize(path) / 1024.0
    except OSError:
        return None

def write_metrics(row: dict, csv_path: Path = METRICS_DIR / "metrics.csv"):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)

def count_loc_from_callable(fn) -> int:
    import inspect, textwrap
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        lines = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
        return len(lines)
    except Exception:
        return None

def runtime_meta():
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
        "plotly": plotly.__version__
    }

In [5]:
from src.data.geo import process_choropleth_data

def load_data(domain: str, dataset_name: str):
    if domain != "geo":
        raise ValueError("Tento notebook je určený pre geografické dáta.")

    df = process_choropleth_data(
        districts_file="okres_2.shp",
        pois_file="pois.csv",
        district_code_col="LAU1_CODE",
        district_name_col="NM3",
        join_predicate="within"
    )

    df["LAU1_CODE"] = df["LAU1_CODE"].astype(str).str.strip()
    df["NM3"] = df["NM3"].astype(str).str.strip()

    return df.sort_values("indicator", ascending=False).reset_index(drop=True)

DF = load_data(DOMAIN, DATASET_NAME)
print(DF.head())
print(DF.shape)

  LAU1_CODE            NM3  poi_count  indicator indicator_label  \
0    SK0102  Bratislava II        232        232       Počet POI   
1    SK0417         Prešov        172        172       Počet POI   
2    SK0233          Nitra        171        171       Počet POI   
3    SK031B         Žilina        140        140       Počet POI   
4    SK0422       Košice I        137        137       Počet POI   

           label                                           geometry  
0  Bratislava II  POLYGON Z ((17.237 48.168 -1.4552e-11, 17.235 ...  
1         Prešov  POLYGON Z ((21.181 49.18 -1.4552e-11, 21.184 4...  
2          Nitra  POLYGON Z ((18.002 48.456 -1.4552e-11, 18.004 ...  
3         Žilina  POLYGON Z ((18.665 49.354 -1.4552e-11, 18.665 ...  
4       Košice I  POLYGON Z ((21.19 48.799 -1.4552e-11, 21.194 4...  
(79, 7)


In [6]:
def plot_baseline(df):
    fig, ax = plt.subplots(figsize=(8, 6))

    df.plot(
        column="indicator",
        cmap="plasma",
        ax=ax,
        legend=False,
        linewidth=0.2,
        edgecolor="black"
    )

    ax.set_title("Počet POI v okresoch Slovenska")
    ax.set_axis_off()

    return fig, "mpl"

In [7]:
def plot_final(df):
    fig, ax = plt.subplots(figsize=(12, 8))

    df.plot(
        column="indicator",
        cmap="plasma",
        linewidth=0.6,
        edgecolor="black",
        legend=True,
        legend_kwds={"label": "Počet POI"},
        ax=ax
    )

    ax.set_title("Tematické mapovanie POI v okresoch Slovenska")
    ax.set_axis_off()

    return fig, "mpl"

In [8]:
def render_and_time(make_fig_callable, df, variant: str, n_runs: int = 5) -> dict:
    times, out_files = [], []

    html_path = None
    png_path = None
    svg_path = None

    # warm-up beh, nezapočítava sa do metrík
    fig, kind = make_fig_callable(df)
    if kind == "mpl":
        plt.close(fig)

    for _ in range(n_runs):
        with Timer() as t:
            fig, kind = make_fig_callable(df)
        times.append(t.elapsed)

        if kind == "mpl":
            plt.close(fig)

    fig, kind = make_fig_callable(df)
    export_elapsed_ms = None

    if kind == "mpl":
        png_path = file_basename("plot", variant, "png")
        with Timer() as t:
            fig.savefig(png_path, dpi=CFG_PLOTS.get("dpi", 120), bbox_inches="tight")
        export_elapsed_ms = t.elapsed
        plt.close(fig)
        out_files.append(png_path)

        if CFG_PLOTS.get("svg", True):
            svg_path = png_path.with_suffix(".svg")
            fig, _ = make_fig_callable(df)
            fig.savefig(svg_path, bbox_inches="tight")
            plt.close(fig)
            out_files.append(svg_path)

    elif kind == "plotly":
        html_path = file_basename("plot", variant, "html")
        with Timer() as t:
            fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
        export_elapsed_ms = t.elapsed
        out_files.append(html_path)

        try:
            png_path = file_basename("plot", variant, "png")
            fig.write_image(png_path, scale=2)
            out_files.append(png_path)

            if CFG_PLOTS.get("svg", True):
                svg_path = file_basename("plot", variant, "svg")
                fig.write_image(svg_path)
                out_files.append(svg_path)
        except Exception as e:
            print("Upozornenie: PNG/SVG export (Plotly) preskočený:", e)

    return {
        "build_ms_median": stats.median(times),
        "build_ms_p95": np.percentile(times, 95),
        "export_ms": export_elapsed_ms,
        "saved": str(html_path or png_path) if (html_path or png_path) else None,
        "html_path": str(html_path) if html_path else None,
        "png_path": str(png_path) if png_path else None,
        "svg_path": str(svg_path) if svg_path else None,
        "html_size_kb": round(file_size_kb(html_path), 2) if html_path else None,
        "png_size_kb": round(file_size_kb(png_path), 2) if png_path else None,
        "svg_size_kb": round(file_size_kb(svg_path), 2) if svg_path else None,
        "all_files": [str(p) for p in out_files]
    }

loc_baseline = count_loc_from_callable(plot_baseline)
loc_final    = count_loc_from_callable(plot_final)

res_baseline = render_and_time(plot_baseline, DF, "baseline", n_runs=N_RUNS)
res_final    = render_and_time(plot_final,    DF, "final",    n_runs=N_RUNS)

meta = runtime_meta()
common = {
    "timestamp": TIMESTAMP,
    "domain": DOMAIN,
    "dataset": DATASET_NAME,
    "library": LIBRARY,
    "task_id": TASK_ID,
    "python": meta["python"],
    "platform": meta["platform"],
    "matplotlib_ver": meta["matplotlib"],
    "seaborn_ver": meta["seaborn"],
    "plotly_ver": meta["plotly"]
}

write_metrics({
    **common,
    "variant": "baseline",
    "build_ms_median": round(res_baseline["build_ms_median"], 2),
    "build_ms_p95": round(res_baseline["build_ms_p95"], 2),
    "export_ms": round(res_baseline["export_ms"] or 0.0, 2),
    "loc": loc_baseline,
    "saved": res_baseline["saved"],
    "html_path": res_baseline["html_path"],
    "png_path": res_baseline["png_path"],
    "svg_path": res_baseline["svg_path"],
    "html_size_kb": res_baseline["html_size_kb"],
    "png_size_kb": res_baseline["png_size_kb"],
    "svg_size_kb": res_baseline["svg_size_kb"]
})

write_metrics({
    **common,
    "variant": "final",
    "build_ms_median": round(res_final["build_ms_median"], 2),
    "build_ms_p95": round(res_final["build_ms_p95"], 2),
    "export_ms": round(res_final["export_ms"] or 0.0, 2),
    "loc": loc_final,
    "saved": res_final["saved"],
    "html_path": res_final["html_path"],
    "png_path": res_final["png_path"],
    "svg_path": res_final["svg_path"],
    "html_size_kb": res_final["html_size_kb"],
    "png_size_kb": res_final["png_size_kb"],
    "svg_size_kb": res_final["svg_size_kb"]
})

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("OUT_ROOT:", OUT_ROOT.resolve())
print("METRICS_DIR:", METRICS_DIR.resolve())
print("Hotovo. Uložené:")
print("  baseline:", res_baseline["saved"])
print("  final   :", res_final["saved"])
print("Metriky ->", METRICS_DIR / "metrics.csv")
print("Metrics exists AFTER:", (METRICS_DIR / "metrics.csv").exists())

PROJECT_ROOT: C:\Users\karol\Bakalarka26\prakticka_cast
OUT_ROOT: C:\Users\karol\Bakalarka26\prakticka_cast\outputs
METRICS_DIR: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics
Hotovo. Uložené:
  baseline: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\geo\geopandas\geo_geopandas_osm_geograficke_data_baseline_2026-04-01_11-05-12.png
  final   : C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\geo\geopandas\geo_geopandas_osm_geograficke_data_final_2026-04-01_11-05-12.png
Metriky -> C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv
Metrics exists AFTER: True


In [9]:
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("OUT_ROOT:", OUT_ROOT.resolve())
print("METRICS_DIR:", METRICS_DIR.resolve())
print("Metrics exists:", (METRICS_DIR / "metrics.csv").exists())

PROJECT_ROOT: C:\Users\karol\Bakalarka26\prakticka_cast
OUT_ROOT: C:\Users\karol\Bakalarka26\prakticka_cast\outputs
METRICS_DIR: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics
Metrics exists: True


In [10]:
m = pd.read_csv(METRICS_DIR / "metrics.csv")

subset = m[
    (m["timestamp"] == TIMESTAMP) &
    (m["domain"] == DOMAIN) &
    (m["dataset"] == DATASET_NAME) &
    (m["library"] == LIBRARY) &
    (m["task_id"] == TASK_ID)
].copy()

display(subset[[
    "variant",
    "build_ms_median",
    "build_ms_p95",
    "export_ms",
    "loc",
    "html_size_kb",
    "png_size_kb",
    "svg_size_kb",
    "saved",
    "html_path",
    "png_path",
    "svg_path"
]])

print("Metrics exists AFTER:", (METRICS_DIR / "metrics.csv").exists())
print("Metrics path:", METRICS_DIR / "metrics.csv")

,variant,build_ms_median,build_ms_p95,export_ms,loc,html_size_kb,png_size_kb,svg_size_kb,saved,html_path,png_path,svg_path
2,baseline,67.54,102.95,74.60,13,NaN,90.34,751.62,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...
3,final,96.01,160.45,135.04,14,NaN,144.46,771.97,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...


Metrics exists AFTER: True
Metrics path: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv
